In [4]:
import duckdb
from pathlib import Path

PROJECT = Path(r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate")
DB_PATH = PROJECT / r"warehouse\realestate.duckdb"
con = duckdb.connect(str(DB_PATH))

y = 2024
t = f"events_311_raw_{y}"

# ensure columns
con.execute(f"ALTER TABLE {t} ADD COLUMN IF NOT EXISTS addr_key VARCHAR")
con.execute(f"ALTER TABLE {t} ADD COLUMN IF NOT EXISTS zip5 VARCHAR")
con.execute(f"ALTER TABLE {t} ADD COLUMN IF NOT EXISTS addr_zip_key VARCHAR")

# zip5: prefer Zip Code column, fallback to Incident Address parsing
con.execute(f"""
UPDATE {t}
SET zip5 = COALESCE(
  NULLIF(REGEXP_EXTRACT(CAST("Zip Code" AS VARCHAR), '(\\d{{5}})', 1), ''),
  NULLIF(REGEXP_EXTRACT("Incident Address", '(\\d{{5}})', 1), '')
)
""")

# addr_key: prefer Incident Street, fallback to Incident Address (street-only)
con.execute(f"""
UPDATE {t}
SET addr_key = TRIM(REGEXP_REPLACE(
    REGEXP_REPLACE(
      UPPER(COALESCE(NULLIF("Incident Street", ''), "Incident Address")),
      '\\s+HOUSTON\\s+TX\\s+\\d{{5}}.*$',
      ''
    ),
    '[^A-Z0-9 ]', ' ', 'g'
))
""")
con.execute(f"UPDATE {t} SET addr_key = TRIM(REGEXP_REPLACE(addr_key, '\\s+', ' ', 'g'))")

# combined
con.execute(f"""
UPDATE {t}
SET addr_zip_key = addr_key || '|' || COALESCE(zip5,'')
""")

# rebuild counts + match
con.execute("""
CREATE OR REPLACE TABLE anchor_zip_key_counts AS
SELECT addr_zip_key, COUNT(DISTINCT acct) AS n_accts
FROM property_anchor
GROUP BY addr_zip_key
""")

con.execute(f"""
CREATE OR REPLACE TABLE property_match_311_{y} AS
SELECT
  e."Case Number" AS case_number,
  e.addr_zip_key,
  p.acct,
  e."Created Date Local" AS created_dt,
  e."Incident Case Type" AS case_type,
  e."Division" AS division
FROM {t} e
JOIN anchor_zip_key_counts kc
  ON e.addr_zip_key = kc.addr_zip_key AND kc.n_accts = 1
JOIN property_anchor p
  ON e.addr_zip_key = p.addr_zip_key
""")

print(con.execute(f"""
WITH c AS (
  SELECT
    (SELECT COUNT(*) FROM {t}) AS total,
    (SELECT COUNT(*) FROM property_match_311_{y}) AS matched
)
SELECT {y} AS year, total, matched, ROUND(matched*1.0/total, 4) AS match_rate
FROM c
""").fetchdf())

con.close()

   year   total  matched  match_rate
0  2024  458818   285415      0.6221


In [5]:
import duckdb
from pathlib import Path

PROJECT = Path(r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate")
DB_PATH = PROJECT / r"warehouse\realestate.duckdb"
years = list(range(2015, 2027))

con = duckdb.connect(str(DB_PATH))

con.execute("""
CREATE OR REPLACE TABLE anchor_zip_key_counts AS
SELECT addr_zip_key, COUNT(DISTINCT acct) AS n_accts
FROM property_anchor
GROUP BY addr_zip_key
""")

for y in years:
    t = f"events_311_raw_{y}"
    print("Year:", y)

    con.execute(f"ALTER TABLE {t} ADD COLUMN IF NOT EXISTS addr_key VARCHAR")
    con.execute(f"ALTER TABLE {t} ADD COLUMN IF NOT EXISTS zip5 VARCHAR")
    con.execute(f"ALTER TABLE {t} ADD COLUMN IF NOT EXISTS addr_zip_key VARCHAR")

    con.execute(f"""
    UPDATE {t}
    SET zip5 = COALESCE(
      NULLIF(REGEXP_EXTRACT(CAST("Zip Code" AS VARCHAR), '(\\d{{5}})', 1), ''),
      NULLIF(REGEXP_EXTRACT("Incident Address", '(\\d{{5}})', 1), '')
    )
    """)

    con.execute(f"""
    UPDATE {t}
    SET addr_key = TRIM(REGEXP_REPLACE(
        REGEXP_REPLACE(
          UPPER(COALESCE(NULLIF("Incident Street", ''), "Incident Address")),
          '\\s+HOUSTON\\s+TX\\s+\\d{{5}}.*$',
          ''
        ),
        '[^A-Z0-9 ]', ' ', 'g'
    ))
    """)
    con.execute(f"UPDATE {t} SET addr_key = TRIM(REGEXP_REPLACE(addr_key, '\\s+', ' ', 'g'))")

    con.execute(f"""
    UPDATE {t}
    SET addr_zip_key = addr_key || '|' || COALESCE(zip5,'')
    """)

    con.execute(f"""
    CREATE OR REPLACE TABLE property_match_311_{y} AS
    SELECT
      e."Case Number" AS case_number,
      e.addr_zip_key,
      p.acct,
      e."Created Date Local" AS created_dt,
      e."Incident Case Type" AS case_type,
      e."Division" AS division
    FROM {t} e
    JOIN anchor_zip_key_counts kc
      ON e.addr_zip_key = kc.addr_zip_key AND kc.n_accts = 1
    JOIN property_anchor p
      ON e.addr_zip_key = p.addr_zip_key
    """)

    con.execute(f"""
    CREATE OR REPLACE TABLE features_311_{y}_by_acct AS
    SELECT
      acct,
      COUNT(*) AS calls_{y},
      COUNT(DISTINCT case_number) AS distinct_cases_{y},
      MAX(created_dt) AS last_311_dt_{y},
      SUM(CASE WHEN UPPER(division) LIKE '%COMMUNITY CODE ENFORCEMENT%' THEN 1 ELSE 0 END) AS code_enforcement_calls_{y},
      SUM(CASE WHEN UPPER(case_type) LIKE '%WEEDS%' OR UPPER(case_type) LIKE '%TRASH%' OR UPPER(case_type) LIKE '%STAGNANT%' THEN 1 ELSE 0 END) AS weeds_trash_calls_{y},
      SUM(CASE WHEN UPPER(case_type) LIKE '%JUNK MOTOR VEHICLE%' THEN 1 ELSE 0 END) AS junk_vehicle_calls_{y}
    FROM property_match_311_{y}
    WHERE created_dt IS NOT NULL
    GROUP BY acct
    """)

    res = con.execute(f"""
    WITH c AS (
      SELECT
        (SELECT COUNT(*) FROM {t}) AS total,
        (SELECT COUNT(*) FROM property_match_311_{y}) AS matched
    )
    SELECT {y} AS year, total, matched, ROUND(matched*1.0/total, 4) AS match_rate
    FROM c
    """).fetchdf()
    print(res)

con.close()
print("Done all years.")

Year: 2015
   year   total  matched  match_rate
0  2015  328580   171661      0.5224
Year: 2016
   year   total  matched  match_rate
0  2016  354118   177249      0.5005
Year: 2017
   year   total  matched  match_rate
0  2017  366444   198575      0.5419
Year: 2018
   year   total  matched  match_rate
0  2018  396405   224155      0.5655
Year: 2019
   year   total  matched  match_rate
0  2019  388295   218702      0.5632
Year: 2020
   year   total  matched  match_rate
0  2020  375122   207716      0.5537
Year: 2021
   year   total  matched  match_rate
0  2021  358917   206970      0.5767
Year: 2022
   year   total  matched  match_rate
0  2022  295647   167386      0.5662
Year: 2023
   year   total  matched  match_rate
0  2023  371499   224679      0.6048
Year: 2024
   year   total  matched  match_rate
0  2024  458818   285415      0.6221
Year: 2025
   year   total  matched  match_rate
0  2025  423060   259421      0.6132
Year: 2026
   year  total  matched  match_rate
0  2026  67571    